<a href="https://colab.research.google.com/github/ANJALICHAMOLI/PyTorch/blob/main/15_pytorch_lstm_next_word_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install nltk

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk

In [ ]:
document = """
Artificial Intelligence and Machine Learning FAQ

What is Artificial Intelligence (AI)?
Artificial Intelligence is the field of computer science focused on creating systems that can perform tasks that typically require human intelligence, such as reasoning, learning, and decision-making.

What is Machine Learning (ML)?
Machine Learning is a subset of AI that enables computers to learn patterns from data without being explicitly programmed.

What is Deep Learning?
Deep Learning is a branch of Machine Learning that uses neural networks with multiple layers to learn complex patterns from large amounts of data.

What is the difference between AI, Machine Learning, and Deep Learning?
AI is the broad field of creating intelligent systems. Machine Learning is a subset of AI that learns from data, while Deep Learning is a subset of Machine Learning that uses deep neural networks.

What programming language is most commonly used for Machine Learning?
Python is the most widely used language because of libraries such as NumPy, Pandas, Scikit-learn, TensorFlow, and PyTorch.

What is supervised learning?
Supervised learning is a type of Machine Learning where models are trained using labeled data containing both inputs and correct outputs.

What is unsupervised learning?
Unsupervised learning involves finding hidden patterns or structures in unlabeled data.

What is reinforcement learning?
Reinforcement learning trains an agent to make decisions by rewarding desirable actions and penalizing undesirable ones.

What is overfitting?
Overfitting occurs when a model learns the training data too well, including noise, resulting in poor performance on new data.

How can overfitting be reduced?
Overfitting can be reduced using techniques such as regularization, dropout, early stopping, cross-validation, and collecting more training data.

What is underfitting?
Underfitting occurs when a model is too simple to capture the underlying patterns in the data.

What is a training dataset?
A training dataset is used to teach a machine learning model by adjusting its parameters.

What is a validation dataset?
A validation dataset is used during training to tune hyperparameters and monitor model performance.

What is a test dataset?
A test dataset is used only after training to evaluate how well the model generalizes to unseen data.

What is an epoch?
An epoch is one complete pass of the entire training dataset through the neural network.

What is batch size?
Batch size is the number of training samples processed before the model updates its parameters.

What is a learning rate?
The learning rate determines how large each parameter update is during optimization.

What is gradient descent?
Gradient descent is an optimization algorithm that minimizes a model's loss by updating parameters in the direction of the negative gradient.

What is backpropagation?
Backpropagation is the algorithm used to compute gradients of the loss with respect to each parameter in a neural network.

What is a loss function?
A loss function measures how far the model's predictions are from the actual target values.

What is cross-entropy loss?
Cross-entropy loss is commonly used for classification problems to measure prediction error.

What is Mean Squared Error (MSE)?
Mean Squared Error is a loss function commonly used in regression problems.

What is a neural network?
A neural network is a computational model made up of layers of interconnected neurons that learn representations from data.

What is an activation function?
An activation function introduces non-linearity into a neural network, enabling it to learn complex relationships.

What is ReLU?
ReLU (Rectified Linear Unit) is an activation function that outputs zero for negative inputs and the input itself for positive values.

What is a CNN?
A Convolutional Neural Network is a deep learning model designed primarily for image and computer vision tasks.

What is an RNN?
A Recurrent Neural Network is designed to process sequential data such as text or time series.

What is an LSTM?
Long Short-Term Memory is a type of RNN that can learn long-term dependencies by using memory cells.

What is a Transformer?
A Transformer is a neural network architecture based on self-attention that efficiently processes sequences in parallel.

What is self-attention?
Self-attention allows each word or token in a sequence to determine how much attention it should pay to every other token.

What is an embedding?
An embedding is a dense numerical vector that represents the meaning of a word, sentence, or other data point.

What is tokenization?
Tokenization is the process of splitting text into smaller units such as words or subwords.

What is a Large Language Model (LLM)?
A Large Language Model is a transformer-based neural network trained on massive amounts of text to understand and generate language.

What is Retrieval-Augmented Generation (RAG)?
RAG combines document retrieval with a language model so responses are grounded in external knowledge.

What is vector embedding?
A vector embedding is a numerical representation of data in a high-dimensional space where similar items are located close together.

What is a vector database?
A vector database stores embeddings and enables efficient similarity search for retrieval systems.

What is cosine similarity?
Cosine similarity measures how similar two vectors are by comparing the angle between them.

What is transfer learning?
Transfer learning uses knowledge learned from one task to improve performance on another related task.

What is fine-tuning?
Fine-tuning is the process of continuing the training of a pretrained model on a specific dataset.

What is PyTorch?
PyTorch is an open-source deep learning framework widely used for research and production.

What is TensorFlow?
TensorFlow is an open-source machine learning framework developed by Google.

What is Scikit-learn?
Scikit-learn is a Python library that provides traditional machine learning algorithms and utilities.

What is model inference?
Inference is the process of using a trained model to make predictions on new data.

What is precision?
Precision measures the proportion of predicted positive samples that are actually positive.

What is recall?
Recall measures the proportion of actual positive samples that are correctly identified.

What is F1-score?
The F1-score is the harmonic mean of precision and recall.

What is accuracy?
Accuracy is the proportion of correct predictions out of all predictions made.

What is confusion matrix?
A confusion matrix summarizes the performance of a classification model by comparing predicted and actual classes.

What is data preprocessing?
Data preprocessing involves cleaning, transforming, and preparing data before training a model.

What is feature engineering?
Feature engineering is the process of creating informative input variables that improve model performance.

What is hyperparameter tuning?
Hyperparameter tuning is the process of finding the best settings for a machine learning model.

Where can beginners learn Machine Learning?
Beginners can start with Python programming, mathematics, Scikit-learn, and then progress to deep learning using TensorFlow or PyTorch.
"""

In [ ]:
# Tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
# tokenize
tokens = word_tokenize(document.lower())

In [ ]:
# build vocab
vocab = {'<unk>':0}

for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token] = len(vocab)

vocab

{'<unk>': 0,
 'artificial': 1,
 'intelligence': 2,
 'and': 3,
 'machine': 4,
 'learning': 5,
 'faq': 6,
 'what': 7,
 'is': 8,
 '(': 9,
 'ai': 10,
 ')': 11,
 '?': 12,
 'the': 13,
 'field': 14,
 'of': 15,
 'computer': 16,
 'science': 17,
 'focused': 18,
 'on': 19,
 'creating': 20,
 'systems': 21,
 'that': 22,
 'can': 23,
 'perform': 24,
 'tasks': 25,
 'typically': 26,
 'require': 27,
 'human': 28,
 ',': 29,
 'such': 30,
 'as': 31,
 'reasoning': 32,
 'decision-making': 33,
 '.': 34,
 'ml': 35,
 'a': 36,
 'subset': 37,
 'enables': 38,
 'computers': 39,
 'to': 40,
 'learn': 41,
 'patterns': 42,
 'from': 43,
 'data': 44,
 'without': 45,
 'being': 46,
 'explicitly': 47,
 'programmed': 48,
 'deep': 49,
 'branch': 50,
 'uses': 51,
 'neural': 52,
 'networks': 53,
 'with': 54,
 'multiple': 55,
 'layers': 56,
 'complex': 57,
 'large': 58,
 'amounts': 59,
 'difference': 60,
 'between': 61,
 'broad': 62,
 'intelligent': 63,
 'learns': 64,
 'while': 65,
 'programming': 66,
 'language': 67,
 'most': 6

In [ ]:
len(vocab)

374

In [ ]:
input_sentences = document.split('\n')

In [ ]:
def text_to_indices(sentence, vocab):

  numerical_sentence = []

  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<unk>'])

  return numerical_sentence


In [ ]:
input_numerical_sentences = []

for sentence in input_sentences:
  input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))


In [ ]:
len(input_numerical_sentences)

159

In [ ]:
#supervised
training_sequence = []
for sentence in input_numerical_sentences:

  for i in range(1, len(sentence)):
    training_sequence.append(sentence[:i+1])

In [ ]:
len(training_sequence)

1124

In [ ]:
#padding
len_list = []

for sequence in training_sequence:
  len_list.append(len(sequence))

max(len_list)

37

In [ ]:
training_sequence[0]

[1, 2]

In [ ]:
#padding
padded_training_sequence = []
for sequence in training_sequence:

  padded_training_sequence.append([0]*(max(len_list) - len(sequence)) + sequence)

In [ ]:
len(padded_training_sequence[10])

37

In [ ]:
padded_training_sequence = torch.tensor(padded_training_sequence, dtype=torch.long)

In [ ]:
padded_training_sequence

tensor([[ 0,  0,  0,  ...,  0,  1,  2],
        [ 0,  0,  0,  ...,  1,  2,  3],
        [ 0,  0,  0,  ...,  2,  3,  4],
        ...,
        [ 0,  0,  0,  ..., 87, 79, 98],
        [ 0,  0,  0,  ..., 79, 98, 80],
        [ 0,  0,  0,  ..., 98, 80, 34]])

In [ ]:
X = padded_training_sequence[:, :-1]
y = padded_training_sequence[:,-1]

In [ ]:
X

tensor([[ 0,  0,  0,  ...,  0,  0,  1],
        [ 0,  0,  0,  ...,  0,  1,  2],
        [ 0,  0,  0,  ...,  1,  2,  3],
        ...,
        [ 0,  0,  0,  ...,  5, 87, 79],
        [ 0,  0,  0,  ..., 87, 79, 98],
        [ 0,  0,  0,  ..., 79, 98, 80]])

In [ ]:
y

tensor([ 2,  3,  4,  ..., 98, 80, 34])

In [ ]:
class CustomDataset(Dataset):

  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [ ]:
dataset = CustomDataset(X,y)

In [ ]:
len(dataset)

1124

In [ ]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
class LSTMModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 100)
    self.lstm = nn.LSTM(100, 150, batch_first=True)
    self.fc = nn.Linear(150, vocab_size)

  def forward(self, x):
    embedded = self.embedding(x)
    intermediate_hidden_states, (final_hidden_state, final_cell_state) = self.lstm(embedded)
    output = self.fc(final_hidden_state.squeeze(0))
    return output

In [ ]:
model = LSTMModel(len(vocab))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model.to(device)

LSTMModel(
  (embedding): Embedding(374, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=374, bias=True)
)

In [ ]:
epochs = 50
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# training loop

for epoch in range(epochs):
  total_loss = 0

  for batch_x, batch_y in dataloader:

    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    optimizer.zero_grad()

    output = model(batch_x)

    loss = criterion(output, batch_y)

    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")

Epoch: 1, Loss: 198.9264
Epoch: 2, Loss: 173.5633
Epoch: 3, Loss: 159.3085
Epoch: 4, Loss: 144.1632
Epoch: 5, Loss: 132.6838
Epoch: 6, Loss: 123.2644
Epoch: 7, Loss: 109.3083
Epoch: 8, Loss: 99.3071
Epoch: 9, Loss: 89.9078
Epoch: 10, Loss: 82.4410
Epoch: 11, Loss: 73.2027
Epoch: 12, Loss: 65.0093
Epoch: 13, Loss: 58.9749
Epoch: 14, Loss: 52.2904
Epoch: 15, Loss: 45.6315
Epoch: 16, Loss: 41.2073
Epoch: 17, Loss: 38.5365
Epoch: 18, Loss: 33.3814
Epoch: 19, Loss: 30.3564
Epoch: 20, Loss: 28.1351
Epoch: 21, Loss: 25.8352
Epoch: 22, Loss: 23.3123
Epoch: 23, Loss: 21.7093
Epoch: 24, Loss: 21.3379
Epoch: 25, Loss: 19.4975
Epoch: 26, Loss: 17.9646
Epoch: 27, Loss: 17.5785
Epoch: 28, Loss: 17.3908
Epoch: 29, Loss: 15.6304
Epoch: 30, Loss: 14.7958
Epoch: 31, Loss: 14.6420
Epoch: 32, Loss: 13.7664
Epoch: 33, Loss: 13.7587
Epoch: 34, Loss: 14.4258
Epoch: 35, Loss: 12.9472
Epoch: 36, Loss: 13.1143
Epoch: 37, Loss: 12.0225
Epoch: 38, Loss: 11.6560
Epoch: 39, Loss: 12.3245
Epoch: 40, Loss: 11.2358
Ep

In [ ]:
# prediction

def prediction(model, vocab, text):

  # tokenize
  tokenized_text = word_tokenize(text.lower())

  # text -> numerical indices
  numerical_text = text_to_indices(tokenized_text, vocab)

  # padding
  padded_text = torch.tensor([0] * (61 - len(numerical_text)) + numerical_text, dtype=torch.long).unsqueeze(0)

  # send to model
  output = model(padded_text)

  # predicted index
  value, index = torch.max(output, dim=1)

  # merge with text
  return text + " " + list(vocab.keys())[index]



In [ ]:
prediction(model, vocab, "what is deep")

'what is deep learning'

In [ ]:
import time

num_tokens = 15
input_text = "deep learning"

for i in range(num_tokens):
  output_text = prediction(model, vocab, input_text)
  print(output_text)
  input_text = output_text
  time.sleep(0.5)


deep learning is
deep learning is a
deep learning is a branch
deep learning is a branch of
deep learning is a branch of machine
deep learning is a branch of machine learning
deep learning is a branch of machine learning that
deep learning is a branch of machine learning that uses
deep learning is a branch of machine learning that uses neural
deep learning is a branch of machine learning that uses neural networks
deep learning is a branch of machine learning that uses neural networks with
deep learning is a branch of machine learning that uses neural networks with multiple
deep learning is a branch of machine learning that uses neural networks with multiple layers
deep learning is a branch of machine learning that uses neural networks with multiple layers to
deep learning is a branch of machine learning that uses neural networks with multiple layers to learn


In [ ]:
dataloader1 = DataLoader(dataset, batch_size=32, shuffle=False)

In [ ]:
# Function to calculate accuracy
def calculate_accuracy(model, dataloader, device):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():  # No need to compute gradients
        for batch_x, batch_y in dataloader1:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            # Get model predictions
            outputs = model(batch_x)

            # Get the predicted word indices
            _, predicted = torch.max(outputs, dim=1)

            # Compare with actual labels
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    accuracy = correct / total * 100
    return accuracy

# Compute accuracy
accuracy = calculate_accuracy(model, dataloader, device)
print(f"Model Accuracy: {accuracy:.2f}%")


Model Accuracy: 93.86%
